# 07. Generación del Dataset POC en Capa Gold (`ds_obras_publicas`)

Construye el dataset plano de prueba de concepto `data/gold/ds_obras_publicas/part-00000.parquet` (**1 fila por `codigo_infobras`**, **37 columnas**) leyendo exclusivamente las 13 tablas Parquet de la capa Silver (`data/silver/`) y la variable de categorías definida en `configs/categoria_paralizacion.yaml`:

- **Grano**: 1 registro por cada obra registrada en `hub_obra_publica` (desde `2023-07-11`).
- **Relaciones vigentes**: `link_obra_entidad`, `link_obra_proyecto`, `link_obra_contratista`, `sat_obra_estado_ejecucion`, `sat_obra_expediente_tecnico`, `sat_proyecto` y `sat_link_obra_contratista` filtrados con `end_date IS NULL`.
- **Roles de contratista**: separación explícita entre ejecución (`rol = 'EJECUCION'` → `ruc_1`, `_eje`) y supervisión (`rol = 'SUPERVISION'` → `ruc_2`, `_sup`).
- **Historial de paralizaciones**: indicador booleano `flg_existe_paralizacion`, `causal_paralizacion_ultimo` del evento más reciente y contadores dinámicos `ind_opa_<categoria>` generados desde `configs/categoria_paralizacion.yaml`.

## 1. Parámetros y rutas

In [ ]:
# ==============================================================================
# PARÁMETROS DE ENTRADA
# ==============================================================================
COMPRESION = "ZSTD"
COMPRESION_NIVEL = 9
ROW_GROUP_SIZE = 100_000
DETENER_ANTE_ERROR = True

# Rutas (None = rutas por defecto del proyecto)
DIR_SILVER = None          # por defecto: data/silver
DIR_DESTINO = None         # por defecto: data/gold
DIR_LOGS = None            # por defecto: data/gold/logs
CONFIG_CAT_PATH = None     # por defecto: configs/categoria_paralizacion.yaml
# ==============================================================================

## 2. Logger y configuración de categorías (`configs/categoria_paralizacion.yaml`)

In [ ]:
import gc
import getpass
import logging
import platform
import shutil
import socket
import sys
import time
import traceback
from datetime import datetime
from pathlib import Path

import duckdb
import pandas as pd
import yaml

pd.set_option("display.max_columns", 60)
pd.set_option("display.max_rows", 200)
pd.set_option("display.width", 250)
pd.set_option("display.max_colwidth", 90)


def obtener_raiz_proyecto() -> Path:
    actual = Path.cwd().resolve()
    for carpeta in [actual, *actual.parents]:
        if (carpeta / "data" / "silver").exists() or (carpeta / "configs").exists():
            return carpeta
    return actual


PROJECT_ROOT = obtener_raiz_proyecto()
SILVER_DIR = Path(DIR_SILVER) if DIR_SILVER else PROJECT_ROOT / "data" / "silver"
GOLD_DIR = Path(DIR_DESTINO) if DIR_DESTINO else PROJECT_ROOT / "data" / "gold"
WORK_DIR = GOLD_DIR / "_work"
TMP_ROOT = WORK_DIR / "_tmp"
OLD_ROOT = WORK_DIR / "_old"
ERROR_ROOT = WORK_DIR / "_error"
LOGS_DIR = Path(DIR_LOGS) if DIR_LOGS else GOLD_DIR / "logs"
RUTA_CONFIG_CAT = Path(CONFIG_CAT_PATH) if CONFIG_CAT_PATH else PROJECT_ROOT / "configs" / "categoria_paralizacion.yaml"

GOLD_DIR.mkdir(parents=True, exist_ok=True)
WORK_DIR.mkdir(parents=True, exist_ok=True)
LOGS_DIR.mkdir(parents=True, exist_ok=True)

assert SILVER_DIR.exists(), f"No existe directorio Silver: {SILVER_DIR}"
assert RUTA_CONFIG_CAT.exists(), f"No existe archivo de configuración de categorías: {RUTA_CONFIG_CAT}"

CFG_CATEGORIA = yaml.safe_load(RUTA_CONFIG_CAT.read_text(encoding="utf-8"))
CATEGORIAS_VALIDAS = [str(c).strip() for c in CFG_CATEGORIA["categories"]]
assert CATEGORIAS_VALIDAS, "categories en categoria_paralizacion.yaml no puede estar vacío"

TABLAS_SILVER_REQUERIDAS = [
    "hub_obra_publica",
    "hub_entidad",
    "hub_proyecto",
    "hub_contratista",
    "link_obra_entidad",
    "link_obra_proyecto",
    "link_obra_contratista",
    "sat_obra_paralizacion",
    "sat_obra_estado_ejecucion",
    "sat_obra_ubigeo",
    "sat_obra_expediente_tecnico",
    "sat_proyecto",
    "sat_link_obra_contratista",
]

RUN_ID = datetime.now().strftime("%Y%m%d_%H%M%S")
LOG_PATH = LOGS_DIR / f"carga_gold_ds_obras_publicas_{RUN_ID}.log"


def configurar_logger() -> logging.Logger:
    """Configura un logger con salida simultánea a archivo y consola."""
    logger = logging.getLogger("carga_gold")
    for h in list(logger.handlers):
        h.close()
        logger.removeHandler(h)
    logger.setLevel(logging.INFO)
    logger.propagate = False
    fmt = logging.Formatter("%(asctime)s [%(levelname)s] %(message)s", datefmt="%Y-%m-%d %H:%M:%S")
    fh = logging.FileHandler(LOG_PATH, encoding="utf-8")
    sh = logging.StreamHandler(sys.stdout)
    for h in (fh, sh):
        h.setFormatter(fmt)
        logger.addHandler(h)
    return logger


log = configurar_logger()

SCHEMA_GOLD = (
    [
        ("codigo_infobras", "VARCHAR"),
        ("codigo_entidad", "VARCHAR"),
        ("nivel_gobierno", "VARCHAR"),
        ("sector_entidad", "VARCHAR"),
        ("codigo_unico", "VARCHAR"),
        ("tipo_formato", "VARCHAR"),
        ("n_obras_relacionadas_proyecto", "INTEGER"),
        ("modalidad_ejecucion_obra", "VARCHAR"),
        ("estado_ejecucion", "VARCHAR"),
        ("naturaleza_obra", "VARCHAR"),
        ("tipo_obra_nivel_1", "VARCHAR"),
        ("tipo_obra_nivel_2", "VARCHAR"),
        ("tipo_obra_nivel_3", "VARCHAR"),
        ("fecha_inicio_obra", "DATE"),
        ("fecha_finalizacion_real", "DATE"),
        ("departamento", "VARCHAR"),
        ("provincia", "VARCHAR"),
        ("distrito", "VARCHAR"),
        ("fecha_aprobacion_expediente", "DATE"),
        ("monto_aprobacion_et", "DOUBLE"),
        ("monto_aprobado_soles", "DOUBLE"),
        ("ruc_1", "VARCHAR"),
        ("razon_social_empresa_consorcio_eje", "VARCHAR"),
        ("monto_contrato_soles_eje", "DOUBLE"),
        ("ruc_2", "VARCHAR"),
        ("razon_social_empresa_consorcio_sup", "VARCHAR"),
        ("monto_contrato_soles_sup", "DOUBLE"),
        ("flg_existe_paralizacion", "BOOLEAN"),
        ("causal_paralizacion_ultimo", "VARCHAR"),
    ]
    + [(f"ind_opa_{c.lower()}", "INTEGER") for c in CATEGORIAS_VALIDAS]
    + [("fecha_foto", "DATE")]
)

log.info("=" * 100)
log.info("INICIO DE CORRIDA GOLD (ds_obras_publicas) | run_id=%s", RUN_ID)
log.info("Usuario=%s | Host=%s | SO=%s", getpass.getuser(), socket.gethostname(), platform.platform())
log.info("Python=%s | pandas=%s | duckdb=%s", platform.python_version(), pd.__version__, duckdb.__version__)
log.info("Origen Silver  : %s", SILVER_DIR)
log.info("Destino Gold   : %s", GOLD_DIR / "ds_obras_publicas")
log.info("Config YAML    : %s (%d categorías -> %d columnas totales)", RUTA_CONFIG_CAT, len(CATEGORIAS_VALIDAS), len(SCHEMA_GOLD))
log.info("Log            : %s", LOG_PATH)

## 3. Ensamblado del dataset (`construir_ds_obras_publicas`)

In [ ]:
def registrar_vistas_silver(con: duckdb.DuckDBPyConnection) -> None:
    """Registra vistas sobre los 13 archivos Parquet de la capa Silver."""
    for tabla in TABLAS_SILVER_REQUERIDAS:
        ruta = SILVER_DIR / tabla / "part-00000.parquet"
        if not ruta.exists():
            raise FileNotFoundError(f"Falta tabla Silver requerida: {ruta}")
        con.execute(f"CREATE OR REPLACE VIEW \"{tabla}\" AS SELECT * FROM read_parquet('{ruta.as_posix()}')")


def construir_ds_obras_publicas(con: duckdb.DuckDBPyConnection, categorias: list[str]) -> None:
    """Ensambla la tabla ds_obras_publicas en memoria leyendo exclusivamente desde las vistas Silver."""
    registrar_vistas_silver(con)

    agg_exprs = ",\n                ".join(
        f"CAST(COALESCE(COUNT(*) FILTER (WHERE flg_existe_paralizacion = TRUE AND categoria_paralizacion = '{cat}'), 0) AS INTEGER) AS \"ind_opa_{cat.lower()}\""
        for cat in categorias
    )
    select_inds = ",\n            ".join(
        f'CAST(COALESCE(pa."ind_opa_{c.lower()}", 0) AS INTEGER) AS "ind_opa_{c.lower()}"'
        for c in categorias
    )

    con.execute(f"""
        CREATE OR REPLACE TABLE ds_obras_publicas AS
        WITH entidad_vigente AS (
            SELECT codigo_infobras, codigo_entidad
            FROM link_obra_entidad
            WHERE end_date IS NULL
        ),
        proyecto_vigente AS (
            SELECT codigo_infobras, codigo_unico
            FROM link_obra_proyecto
            WHERE end_date IS NULL
        ),
        eje_vigente AS (
            SELECT
                l.codigo_infobras,
                l.ruc AS ruc_1,
                hc.razon_social_empresa_consorcio AS razon_social_empresa_consorcio_eje,
                sc.monto_contrato_soles AS monto_contrato_soles_eje
            FROM link_obra_contratista l
            LEFT JOIN hub_contratista hc ON hc.ruc = l.ruc
            LEFT JOIN sat_link_obra_contratista sc
                   ON sc.codigo_infobras = l.codigo_infobras
                  AND sc.rol = l.rol
                  AND sc.ruc = l.ruc
                  AND sc.end_date IS NULL
            WHERE l.end_date IS NULL AND l.rol = 'EJECUCION'
        ),
        sup_vigente AS (
            SELECT
                l.codigo_infobras,
                l.ruc AS ruc_2,
                hc.razon_social_empresa_consorcio AS razon_social_empresa_consorcio_sup,
                sc.monto_contrato_soles AS monto_contrato_soles_sup
            FROM link_obra_contratista l
            LEFT JOIN hub_contratista hc ON hc.ruc = l.ruc
            LEFT JOIN sat_link_obra_contratista sc
                   ON sc.codigo_infobras = l.codigo_infobras
                  AND sc.rol = l.rol
                  AND sc.ruc = l.ruc
                  AND sc.end_date IS NULL
            WHERE l.end_date IS NULL AND l.rol = 'SUPERVISION'
        ),
        ultima_causal AS (
            SELECT codigo_infobras, causal_paralizacion AS causal_paralizacion_ultimo
            FROM sat_obra_paralizacion
            WHERE flg_existe_paralizacion = TRUE
            QUALIFY ROW_NUMBER() OVER (
                PARTITION BY codigo_infobras
                ORDER BY start_date DESC, fecha_foto DESC
            ) = 1
        ),
        paralizaciones_agg AS (
            SELECT
                codigo_infobras,
                COALESCE(BOOL_OR(flg_existe_paralizacion), FALSE) AS flg_existe_paralizacion,
                {agg_exprs}
            FROM sat_obra_paralizacion
            GROUP BY codigo_infobras
        )
        SELECT
            CAST(h.codigo_infobras AS VARCHAR) AS codigo_infobras,
            CAST(ev.codigo_entidad AS VARCHAR) AS codigo_entidad,
            CAST(he.nivel_gobierno AS VARCHAR) AS nivel_gobierno,
            CAST(he.sector_entidad AS VARCHAR) AS sector_entidad,
            CAST(pv.codigo_unico AS VARCHAR) AS codigo_unico,
            CAST(hp.tipo_formato AS VARCHAR) AS tipo_formato,
            CAST(sp.n_obras_relacionadas_proyecto AS INTEGER) AS n_obras_relacionadas_proyecto,
            CAST(h.modalidad_ejecucion_obra AS VARCHAR) AS modalidad_ejecucion_obra,
            CAST(see.estado_ejecucion AS VARCHAR) AS estado_ejecucion,
            CAST(h.naturaleza_obra AS VARCHAR) AS naturaleza_obra,
            CAST(h.tipo_obra_nivel_1 AS VARCHAR) AS tipo_obra_nivel_1,
            CAST(h.tipo_obra_nivel_2 AS VARCHAR) AS tipo_obra_nivel_2,
            CAST(h.tipo_obra_nivel_3 AS VARCHAR) AS tipo_obra_nivel_3,
            CAST(h.fecha_inicio_obra AS DATE) AS fecha_inicio_obra,
            CAST(h.fecha_finalizacion_real AS DATE) AS fecha_finalizacion_real,
            CAST(su.departamento AS VARCHAR) AS departamento,
            CAST(su.provincia AS VARCHAR) AS provincia,
            CAST(su.distrito AS VARCHAR) AS distrito,
            CAST(setec.fecha_aprobacion_expediente AS DATE) AS fecha_aprobacion_expediente,
            CAST(setec.monto_aprobacion_et AS DOUBLE) AS monto_aprobacion_et,
            CAST(setec.monto_aprobado_soles AS DOUBLE) AS monto_aprobado_soles,
            CAST(ej.ruc_1 AS VARCHAR) AS ruc_1,
            CAST(ej.razon_social_empresa_consorcio_eje AS VARCHAR) AS razon_social_empresa_consorcio_eje,
            CAST(ej.monto_contrato_soles_eje AS DOUBLE) AS monto_contrato_soles_eje,
            CAST(sv.ruc_2 AS VARCHAR) AS ruc_2,
            CAST(sv.razon_social_empresa_consorcio_sup AS VARCHAR) AS razon_social_empresa_consorcio_sup,
            CAST(sv.monto_contrato_soles_sup AS DOUBLE) AS monto_contrato_soles_sup,
            CAST(COALESCE(pa.flg_existe_paralizacion, FALSE) AS BOOLEAN) AS flg_existe_paralizacion,
            CAST(uc.causal_paralizacion_ultimo AS VARCHAR) AS causal_paralizacion_ultimo,
            {select_inds},
            CAST(h.fecha_foto AS DATE) AS fecha_foto
        FROM hub_obra_publica h
        LEFT JOIN entidad_vigente ev ON ev.codigo_infobras = h.codigo_infobras
        LEFT JOIN hub_entidad he ON he.codigo_entidad = ev.codigo_entidad
        LEFT JOIN proyecto_vigente pv ON pv.codigo_infobras = h.codigo_infobras
        LEFT JOIN hub_proyecto hp ON hp.codigo_unico = pv.codigo_unico
        LEFT JOIN sat_proyecto sp ON sp.codigo_unico = pv.codigo_unico AND sp.end_date IS NULL
        LEFT JOIN sat_obra_estado_ejecucion see ON see.codigo_infobras = h.codigo_infobras AND see.end_date IS NULL
        LEFT JOIN sat_obra_ubigeo su ON su.codigo_infobras = h.codigo_infobras
        LEFT JOIN sat_obra_expediente_tecnico setec ON setec.codigo_infobras = h.codigo_infobras AND setec.end_date IS NULL
        LEFT JOIN eje_vigente ej ON ej.codigo_infobras = h.codigo_infobras
        LEFT JOIN sup_vigente sv ON sv.codigo_infobras = h.codigo_infobras
        LEFT JOIN paralizaciones_agg pa ON pa.codigo_infobras = h.codigo_infobras
        LEFT JOIN ultima_causal uc ON uc.codigo_infobras = h.codigo_infobras
    """)

## 4. Escritura temporal (`escribir_gold_tmp`)

In [ ]:
def escribir_gold_tmp(con: duckdb.DuckDBPyConnection) -> Path:
    """Exporta ds_obras_publicas ordenado por codigo_infobras hacia _work/_tmp/ds_obras_publicas/."""
    if TMP_ROOT.exists():
        shutil.rmtree(TMP_ROOT)
    carpeta = TMP_ROOT / "ds_obras_publicas"
    carpeta.mkdir(parents=True, exist_ok=True)
    archivo = carpeta / "part-00000.parquet"
    t0 = time.time()
    con.execute(
        f"COPY (SELECT * FROM ds_obras_publicas ORDER BY codigo_infobras) "
        f"TO '{archivo.as_posix()}' "
        f"(FORMAT PARQUET, COMPRESSION {COMPRESION}, COMPRESSION_LEVEL {int(COMPRESION_NIVEL)}, ROW_GROUP_SIZE {int(ROW_GROUP_SIZE)})"
    ).fetchall()
    n_rows = con.execute("SELECT count(*) FROM ds_obras_publicas").fetchall()[0][0]
    tam_mb = archivo.stat().st_size / 1024**2
    log.info("[_work] ds_obras_publicas escrito: %10s filas | %6.2f MB | %.2f s", f"{n_rows:,}", tam_mb, time.time() - t0)
    return archivo

## 5. Validaciones bloqueantes (`validar_gold`: `G01`–`G05`)

In [ ]:
def _scalar(con: duckdb.DuckDBPyConnection, sql: str):
    return con.execute(sql).fetchall()[0][0]


def validar_gold(con_mem: duckdb.DuckDBPyConnection, ruta_tmp: Path, categorias: list[str]) -> pd.DataFrame:
    """Ejecuta validaciones G01-G05 sobre el Parquet temporal antes de promover a data/gold/."""
    checks = []

    def chk(id_, descripcion, esperado, obtenido, ok):
        checks.append(
            {
                "id": id_,
                "tabla": "ds_obras_publicas",
                "validacion": descripcion,
                "esperado": str(esperado),
                "obtenido": str(obtenido),
                "estado": "OK" if ok else "FALLA",
            }
        )

    assert ruta_tmp.exists() and ruta_tmp.stat().st_size > 0, f"Parquet temporal vacío o inexistente: {ruta_tmp}"

    con_v = duckdb.connect()
    try:
        ruta_hub = (SILVER_DIR / "hub_obra_publica" / "part-00000.parquet").as_posix()
        ruta_sat_p = (SILVER_DIR / "sat_obra_paralizacion" / "part-00000.parquet").as_posix()
        con_v.execute(f"CREATE VIEW gold_tmp AS SELECT * FROM read_parquet('{ruta_tmp.as_posix()}')")
        con_v.execute(f"CREATE VIEW silver_hub AS SELECT * FROM read_parquet('{ruta_hub}')")
        con_v.execute(f"CREATE VIEW silver_sat_p AS SELECT * FROM read_parquet('{ruta_sat_p}')")

        # G01: 1 fila por codigo_infobras y paridad exacta con hub_obra_publica
        n_gold, n_dist, n_null_pk = con_v.execute(
            "SELECT count(*), count(DISTINCT codigo_infobras), count(*) FILTER (WHERE codigo_infobras IS NULL) FROM gold_tmp"
        ).fetchall()[0]
        n_hub = int(_scalar(con_v, "SELECT count(*) FROM silver_hub"))
        chk(
            "G01",
            "1 fila por codigo_infobras = count(hub_obra_publica)",
            f"{n_hub:,} filas únicas, 0 nulos",
            f"gold={n_gold:,}, distinct={n_dist:,}, nulls={n_null_pk}",
            n_gold == n_dist and n_null_pk == 0 and n_gold == n_hub,
        )

        # G02: Esquema y tipos exactos de las columnas
        desc = [(r[0], r[1]) for r in con_v.execute("DESCRIBE gold_tmp").fetchall()]
        chk(
            "G02",
            f"Esquema y tipos exactos ({len(SCHEMA_GOLD)} columnas)",
            f"{len(SCHEMA_GOLD)} cols coincidentes",
            f"{len(desc)} cols (schema_ok={desc == SCHEMA_GOLD})",
            desc == SCHEMA_GOLD,
        )

        # G03: Cero nulos y cero negativos en ind_opa_*
        ind_cols = [f"ind_opa_{c.lower()}" for c in categorias]
        cond_g03 = " OR ".join(f'"{c}" IS NULL OR "{c}" < 0' for c in ind_cols)
        n_err_ind = int(_scalar(con_v, f"SELECT count(*) FROM gold_tmp WHERE {cond_g03}"))
        chk("G03", "Cero nulos y cero negativos (< 0) en ind_opa_*", 0, n_err_ind, n_err_ind == 0)

        # G04: Conservación de eventos respecto a sat_obra_paralizacion
        suma_inds_expr = " + ".join(f'CAST("{c}" AS BIGINT)' for c in ind_cols)
        tot_gold_eventos = int(_scalar(con_v, f"SELECT COALESCE(SUM({suma_inds_expr}), 0) FROM gold_tmp"))
        tot_silver_eventos = int(_scalar(con_v, "SELECT count(*) FILTER (WHERE flg_existe_paralizacion = TRUE) FROM silver_sat_p"))
        chk(
            "G04",
            "Conservación de eventos: SUM(ind_opa_*) = eventos flg=TRUE en sat_obra_paralizacion",
            f"{tot_silver_eventos:,}",
            f"{tot_gold_eventos:,}",
            tot_gold_eventos == tot_silver_eventos,
        )

        # G05: Consistencia de flg_existe_paralizacion y causal_paralizacion_ultimo
        n_err_g05 = int(_scalar(
            con_v,
            f"""
            SELECT count(*) FROM gold_tmp
            WHERE flg_existe_paralizacion IS NULL
               OR flg_existe_paralizacion != (({suma_inds_expr}) > 0)
               OR (flg_existe_paralizacion = FALSE AND causal_paralizacion_ultimo IS NOT NULL)
            """,
        ))
        chk(
            "G05",
            "Consistencia de flg_existe_paralizacion, suma ind_opa_* > 0 y causal_paralizacion_ultimo",
            0, n_err_g05, n_err_g05 == 0,
        )
    finally:
        con_v.close()

    return pd.DataFrame(checks)

## 6. Promoción atómica (`promover_gold`)

In [ ]:
def promover_gold(ruta_tmp: Path) -> Path:
    """Promueve atómicamente data/gold/_work/_tmp/ds_obras_publicas hacia data/gold/ds_obras_publicas/ (con rollback)."""
    if OLD_ROOT.exists():
        shutil.rmtree(OLD_ROOT)
    OLD_ROOT.mkdir(parents=True, exist_ok=True)
    destino = GOLD_DIR / "ds_obras_publicas"
    respaldo = OLD_ROOT / "ds_obras_publicas"
    origen = ruta_tmp.parent
    movido_old = False
    promovido = False
    try:
        if destino.exists():
            destino.rename(respaldo)
            movido_old = True
        origen.rename(destino)
        promovido = True
    except Exception:
        if promovido and destino.exists():
            shutil.rmtree(destino, ignore_errors=True)
        if movido_old and respaldo.exists() and not destino.exists():
            respaldo.rename(destino)
        raise
    shutil.rmtree(OLD_ROOT, ignore_errors=True)
    shutil.rmtree(TMP_ROOT, ignore_errors=True)
    return destino / "part-00000.parquet"

## 7. Ejecución del proceso Gold y resumen estadístico

In [ ]:
con = duckdb.connect(database=":memory:")
t_total = time.time()
ruta_final = None
df_validacion = pd.DataFrame()
df_dist_flg = pd.DataFrame()
df_resumen_ind = pd.DataFrame()
df_nulos = pd.DataFrame()

try:
    t0 = time.time()
    construir_ds_obras_publicas(con, CATEGORIAS_VALIDAS)
    n_filas = int(con.execute("SELECT count(*) FROM ds_obras_publicas").fetchall()[0][0])
    log.info("Dataset ensamblado en memoria: %s filas x %d columnas (%.2f s)", f"{n_filas:,}", len(SCHEMA_GOLD), time.time() - t0)

    ruta_tmp = escribir_gold_tmp(con)
    df_validacion = validar_gold(con, ruta_tmp, CATEGORIAS_VALIDAS)
    for r in df_validacion.itertuples():
        (log.info if r.estado == "OK" else log.error)(
            "[VALIDACION] %s | %s | esperado=%s | obtenido=%s -> %s",
            r.id, r.validacion, r.esperado, r.obtenido, r.estado,
        )

    n_fallas = int((df_validacion["estado"] == "FALLA").sum())
    if n_fallas > 0:
        err_dir = ERROR_ROOT / RUN_ID
        err_dir.mkdir(parents=True, exist_ok=True)
        if TMP_ROOT.exists():
            TMP_ROOT.rename(err_dir / "_tmp")
        raise RuntimeError(f"Se detectaron {n_fallas} falla(s) en validaciones Gold; artefactos movidos a {err_dir}")

    ruta_final = promover_gold(ruta_tmp)
    log.info("Promoción atómica completada: %s (%.2f MB)", ruta_final, ruta_final.stat().st_size / 1024**2)

    df_dist_flg = con.execute("""
        SELECT
            flg_existe_paralizacion,
            count(*) AS obras,
            round(100.0 * count(*) / SUM(count(*)) OVER (), 2) AS pct
        FROM ds_obras_publicas
        GROUP BY 1
        ORDER BY 1 DESC
    """).df()

    ind_rows = []
    for c in CATEGORIAS_VALIDAS:
        col = f"ind_opa_{c.lower()}"
        tot_ev, n_obras_pos, max_ev = con.execute(
            f'SELECT COALESCE(SUM("{col}"), 0), count(*) FILTER (WHERE "{col}" > 0), COALESCE(MAX("{col}"), 0) FROM ds_obras_publicas'
        ).fetchall()[0]
        ind_rows.append({"columna": col, "total_eventos": int(tot_ev), "obras_con_eventos": int(n_obras_pos), "max_por_obra": int(max_ev)})
    df_resumen_ind = pd.DataFrame(ind_rows)

    nulo_rows = []
    for col_name, col_type in SCHEMA_GOLD:
        n_null = int(con.execute(f'SELECT count(*) FILTER (WHERE "{col_name}" IS NULL) FROM ds_obras_publicas').fetchall()[0][0])
        nulo_rows.append({
            "columna": col_name,
            "tipo": col_type,
            "nulos": n_null,
            "no_nulos": n_filas - n_null,
            "pct_completitud": round(100.0 * (n_filas - n_null) / max(n_filas, 1), 2),
        })
    df_nulos = pd.DataFrame(nulo_rows)
except Exception as exc:
    log.error("ERROR EN CORRIDA GOLD: %s: %s\n%s", type(exc).__name__, exc, traceback.format_exc())
    if DETENER_ANTE_ERROR:
        raise
finally:
    con.close()
    gc.collect()
    log.info("FIN DE CORRIDA GOLD | run_id=%s | duración=%.2f s", RUN_ID, time.time() - t_total)
    log.info("=" * 100)

display(df_validacion)
display(df_dist_flg)
display(df_resumen_ind)
display(df_nulos)

## 8. Auditoría sobre obras de control

In [ ]:
OBRAS_CONTROL = ["501650", "540677", "166922"]
if ruta_final and ruta_final.exists():
    con_q = duckdb.connect()
    try:
        placeholders = ", ".join(f"'{c}'" for c in OBRAS_CONTROL)
        df_ctrl = con_q.execute(
            f"SELECT * FROM read_parquet('{ruta_final.as_posix()}') "
            f"WHERE codigo_infobras IN ({placeholders}) ORDER BY codigo_infobras"
        ).df()
        display(df_ctrl)
    finally:
        con_q.close()
else:
    print("Aún no se generó", GOLD_DIR / "ds_obras_publicas")